# DuckDB Main Project — Production Refresh (v2: Fast + Safe)

Daily **Excel → DuckDB** refresh engine. Same behaviour and same table schema as the previous version — only much faster.

### Absolute rule: source files are NEVER modified
The notebook only **reads** the Excel files (`stat`, read-only `zipfile`, DuckDB `read_xlsx`, and an optional read-only copy).
It never writes, renames, moves or deletes anything in the source folders. Every run takes a before/after fingerprint
(modified-time + size) of all source files and reports `Source integrity: OK`.

### What changed (speed)
| Before | Now |
|---|---|
| Row-by-row Python loop per cell | **One SQL statement per file** (DuckDB engine) |
| `executemany` — 1 INSERT per row | **Bulk insert** from a DataFrame |
| Month filter *after* building all rows | **Month filter inside SQL** — unwanted rows never leave DuckDB |
| File opened twice (openpyxl + read) | Sheet name read from the zip index; file **read once** |
| Files processed one at a time | **Parallel reads** (threads), single writer |
| 1 tracker query per file | Tracker loaded **once** into a dict |

### What changed (safety)
- **Date parsing is format-tolerant** (ISO, `dd/mm/yyyy`, `mm/dd/yyyy`, `04-Sep-2026`, `Sep 4, 2026`, with/without time, Excel serials, 2-digit years …).
- Dates are used **only to choose the month** — the original text is stored untouched (all columns stay `VARCHAR`).
- Rows whose date cannot be understood are **never silently dropped**: they are counted, logged, and the exact bad values are saved in `Bad_Date_Log`.
- Slash-dates that would land in a *different month* if read the other way round (`04/09` vs `09/04`) are counted as **ambiguous** and flagged.
- Change detection uses **modified-time + file size**.
- `verify_month()` reconciles the log against what is really in the tables.


In [1]:
import re
import time
import shutil
import tempfile
import zipfile
import xml.etree.ElementTree as ET

from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, date
from pathlib import Path

import duckdb
import pandas as pd


In [2]:
# ==========================================
# PROJECT CONFIGURATION
# ==========================================

PROJECT_FOLDER = Path(
    r"C:\Users\Abhay\OneDrive\Desktop\Claude DuckDB result"
)

# >>> Change ONLY this one value when moving to another month. <<<
# Expected structure:  ...\DuckDB\<YEAR>\<MonthName>   e.g. ...\DuckDB\2026\Sep
SOURCE_MONTH_FOLDER = Path(
    r"C:\Users\Abhay\OneDrive\Desktop\DuckDB\2026\Sep"
)

# CURRENT_MONTH ("Sep26") is derived from the folder path. Set an override only if
# your folders do not follow the <YEAR>\<MonthName> pattern.
CURRENT_MONTH_OVERRIDE = None

# ---- Performance / behaviour switches ----
MAX_WORKERS = 4            # parallel file readers (try 4-8 on a network drive)
COPY_TO_LOCAL = False      # True = copy each file to a temp folder first (read-only on source);
                           # can help on a slow network drive.
DAY_FIRST = True           # How to read ambiguous dates like 04/09/2026  (True = 4 Sep, India style)
BAD_DATE_SAMPLE_LIMIT = 50 # max distinct unreadable date values saved per file

MONTHS = {"Jan":1, "Feb":2, "Mar":3, "Apr":4, "May":5, "Jun":6,
          "Jul":7, "Aug":8, "Sep":9, "Oct":10, "Nov":11, "Dec":12}

if CURRENT_MONTH_OVERRIDE:
    CURRENT_MONTH = CURRENT_MONTH_OVERRIDE
else:
    _mon = SOURCE_MONTH_FOLDER.name.strip()[:3].title()
    _yr = SOURCE_MONTH_FOLDER.parent.name.strip()
    if _mon not in MONTHS or not re.fullmatch(r"\d{4}", _yr):
        raise ValueError(
            f"Cannot derive month from folder '{SOURCE_MONTH_FOLDER}'. "
            "Use <YEAR>\\<MonthName> folders or set CURRENT_MONTH_OVERRIDE (e.g. 'Sep26')."
        )
    CURRENT_MONTH = f"{_mon}{_yr[-2:]}"

TARGET_YEAR = int("20" + CURRENT_MONTH[-2:])
TARGET_MONTH = MONTHS[CURRENT_MONTH[:3]]
MONTH_START = date(TARGET_YEAR, TARGET_MONTH, 1)
MONTH_END = date(TARGET_YEAR + (TARGET_MONTH == 12), TARGET_MONTH % 12 + 1, 1)   # first day of NEXT month

LIVE_FOLDER = SOURCE_MONTH_FOLDER / "LIVE"
WV_FOLDER = SOURCE_MONTH_FOLDER / "WV"

DB_FILE = PROJECT_FOLDER / "company_data.duckdb"

LIVE_TABLE = f"LIVE_{CURRENT_MONTH}"
WV_TABLE = f"WV_{CURRENT_MONTH}"

print("Project folder :", PROJECT_FOLDER)
print("Source folder  :", SOURCE_MONTH_FOLDER)
print("Month          :", CURRENT_MONTH, f"({MONTH_START} to {MONTH_END}, end exclusive)")
print("LIVE table     :", LIVE_TABLE)
print("WV table       :", WV_TABLE)
print("Database       :", DB_FILE)
print("Date order     :", "day-first" if DAY_FIRST else "month-first", "(for ambiguous dates)")


Project folder : C:\Users\Abhay\OneDrive\Desktop\Claude DuckDB result
Source folder  : C:\Users\Abhay\OneDrive\Desktop\DuckDB\2026\Sep
Month          : Sep26 (2026-09-01 to 2026-10-01, end exclusive)
LIVE table     : LIVE_Sep26
WV table       : WV_Sep26
Database       : C:\Users\Abhay\OneDrive\Desktop\Claude DuckDB result\company_data.duckdb
Date order     : day-first (for ambiguous dates)


In [3]:
# ==========================================
# SOURCE FOLDER VALIDATION
# ==========================================

for folder, label in [
    (SOURCE_MONTH_FOLDER, "Source month"),
    (LIVE_FOLDER, "LIVE"),
    (WV_FOLDER, "WV"),
]:
    if not folder.exists():
        raise FileNotFoundError(f"{label} folder not found: {folder}")

PROJECT_FOLDER.mkdir(parents=True, exist_ok=True)   # project folder is OUR folder, not the source

print("All source folders are available.")


All source folders are available.


In [4]:
# ==========================================
# DUCKDB CONNECTION
# ==========================================

con = duckdb.connect(str(DB_FILE))
print("Connected:", DB_FILE)


Connected: C:\Users\Abhay\OneDrive\Desktop\Claude DuckDB result\company_data.duckdb


In [5]:
# ==========================================
# STANDARD SCHEMAS  (unchanged)
# ==========================================

live_required_columns = ['Date Created', 'Date Modified', 'Lead Type', 'RA Name 1', 'RA Name 2', 'Client Name', 'Campaign Code', 'Asset Title', 'Salutation', 'First Name', 'Last Name', 'Email ID', 'Job Title', 'Job Title Level', 'Job Title Department', 'Company Name', 'Industry Type', 'Standard Industry', 'Address 1', 'City', 'State', 'Zip Code', 'Country', 'Address Type', 'Phone Number', 'Direct Number', 'Exact Employee Size', 'Employee Size Category', 'Revenue Size', 'Revenue Size Category', 'Job Title Link', 'Job Title Link - Salesnav', 'Industry Type Link', 'Employee Size Link', 'Revenue Size Link', 'Website', 'Disposition', 'RA 1 Remarks', 'RA 2 Remarks', 'QA Status', 'Primary Reason', 'Secondary reason', 'QA Comments', 'Special Comments', 'Auditor', 'Domain', 'Status', 'ATA', 'Million Verifier', 'EmailListVerify', 'QA Confidence Score', 'Sr No.']

wv_required_columns = ['VR Status', 'Status', 'Date', 'Lead Type', 'Client Name', 'Campaign Code', 'Campaign Name', 'Salutation', 'First Name', 'Last Name', 'Email ID', 'Job Title', 'Job Title Level', 'Job Title Department', 'Company Name', 'Industry Type', 'Standard Industry', 'Address 1', 'City', 'State', 'Zip Code', 'Country', 'Address Type', 'Phone Number', 'Direct Number', 'Exact Employee Size', 'Employee Size Category', 'Revenue Size', 'Revenue Size Category', 'Job Title Link', 'Job Title Link  - Salesnav', 'Industry Type Link', 'Employee Size Link', 'Revenue Size Link', 'Website', 'Disposition', 'RA 1 Remarks', 'RA 2 Remarks', 'QA Status', 'Primary Reason', 'Secondary reason', 'QA Comments', 'Special Comments', 'Auditor', 'Domain', 'Status_1', 'ATA', 'Million Verifier', 'EmailListVerify', 'QA Confidence Score']

SOURCE_METADATA_COLUMNS = ["Campaign_ID", "Source_File", "Source_Path"]

LIVE_CQ_COLUMNS = [f"CQ{i}" for i in range(1, 11)]
WV_CQ_COLUMNS = [f"CQ{i}" for i in range(1, 11)]
LIVE_STANDARD_COLUMNS = live_required_columns.copy()
WV_STANDARD_COLUMNS = wv_required_columns.copy()

COLUMN_ALIASES = {"salesnav job title link": "Job Title Link - Salesnav"}

assert len(live_required_columns) == 52
assert len(wv_required_columns) == 50

LIVE_TABLE_COLUMNS = live_required_columns + LIVE_CQ_COLUMNS + SOURCE_METADATA_COLUMNS
WV_TABLE_COLUMNS = wv_required_columns + WV_CQ_COLUMNS + SOURCE_METADATA_COLUMNS

print("LIVE standard columns:", len(live_required_columns))
print("WV standard columns  :", len(wv_required_columns))


LIVE standard columns: 52
WV standard columns  : 50


In [6]:
# ==========================================
# HEADER NORMALIZATION + COLUMN PLANNING
# ==========================================
# Header-matching rules are IDENTICAL to the previous version.
# The only difference: instead of building rows one by one in Python, we turn the
# result into a small "plan" (which source column feeds which standard column) and
# let SQL apply it to all rows at once.

def normalize_header(header):
    header = str(header).strip().lower()
    header = re.sub(r"[_-]+", " ", header)
    header = re.sub(r"[^\w\s]", "", header)
    header = re.sub(r"\s+", " ", header)
    return header.strip()


def build_live_header_results(source_headers):
    """LIVE: columns are identified by POSITION (52-column schema)."""
    header_results = {}
    for position, expected in enumerate(LIVE_STANDARD_COLUMNS, start=1):
        if position > len(source_headers):
            header_results[position] = {"type": "unidentified", "expected": expected}
            continue
        actual = source_headers[position - 1]
        if actual == expected:
            header_results[position] = {"type": "standard", "column": expected}
            continue
        if normalize_header(actual) == normalize_header(expected):
            header_results[position] = {"type": "standard", "column": expected}
            continue
        if COLUMN_ALIASES.get(normalize_header(actual)) == expected:
            header_results[position] = {"type": "standard", "column": expected}
            continue
        header_results[position] = {"type": "unidentified", "expected": expected, "actual": actual}

    for position in range(len(LIVE_STANDARD_COLUMNS) + 1, len(source_headers) + 1):
        header_results[position] = {"type": "cq", "actual": source_headers[position - 1]}
    return header_results


def build_wv_header_results(source_headers):
    """WV: columns are identified by HEADER NAME, not position."""
    header_results = {}
    normalized_expected = {normalize_header(c): c for c in WV_STANDARD_COLUMNS}
    wv_aliases = {"salesnav job title link": "Job Title Link  - Salesnav"}

    for position, actual in enumerate(source_headers, start=1):
        if actual in WV_STANDARD_COLUMNS:
            header_results[position] = {"type": "standard", "column": actual}
            continue
        normalized_actual = normalize_header(actual)
        standard_match = normalized_expected.get(normalized_actual)
        if standard_match is not None:
            header_results[position] = {"type": "standard", "column": standard_match}
            continue
        alias_match = wv_aliases.get(normalized_actual)
        if alias_match in WV_STANDARD_COLUMNS:
            header_results[position] = {"type": "standard", "column": alias_match}
            continue
        header_results[position] = {"type": "cq", "actual": actual}
    return header_results


def results_to_plan(source_headers, header_results):
    """
    mapping  : standard column -> source header   (if two source columns map to the same
               standard column, the LAST one wins — same as the previous version)
    cq_items : [(source header, text prefix)] for every non-standard column
    """
    mapping, cq_items = {}, []
    for position, result in sorted(header_results.items()):
        if position > len(source_headers):
            continue
        src = source_headers[position - 1]
        if result["type"] == "standard":
            mapping[result["column"]] = src
        elif result["type"] == "unidentified":
            cq_items.append((src, f"At {position}: "))
        elif result["type"] == "cq":
            cq_items.append((src, "CQ: "))
    return mapping, cq_items


In [7]:
# ==========================================
# TABLES: MONTHLY + TRACKER + PROCESS LOG + BAD-DATE LOG
# ==========================================

def ensure_columns(table_name, wanted):
    """wanted = {column: sql_type}. Adds anything missing (safe upgrade of old DBs)."""
    existing = {
        r[0] for r in con.execute(
            "SELECT column_name FROM information_schema.columns WHERE table_name = ?",
            [table_name],
        ).fetchall()
    }
    for column, sql_type in wanted.items():
        if column not in existing:
            con.execute(f'ALTER TABLE "{table_name}" ADD COLUMN "{column}" {sql_type}')


def ensure_monthly_table(table_name, columns):
    exists = con.execute(
        "SELECT count(*) FROM information_schema.tables WHERE table_name = ?", [table_name]
    ).fetchone()[0]
    if exists:
        ensure_columns(table_name, {c: "VARCHAR" for c in columns})
        return
    column_sql = ",\n".join(f'"{c}" VARCHAR' for c in columns)
    con.execute(f'CREATE TABLE "{table_name}" (\n{column_sql}\n)')


ensure_monthly_table(LIVE_TABLE, LIVE_TABLE_COLUMNS)
ensure_monthly_table(WV_TABLE, WV_TABLE_COLUMNS)

# ---- File tracker ----
con.execute("""
CREATE TABLE IF NOT EXISTS File_Tracker (
    Source_Path VARCHAR, Source_File VARCHAR, File_Type VARCHAR, Campaign_ID VARCHAR,
    Month VARCHAR, Last_Modified TIMESTAMP, Last_Run_ID VARCHAR, Last_Status VARCHAR
)""")
ensure_columns("File_Tracker", {"Last_Size": "BIGINT"})

# ---- Process log ----
con.execute("""
CREATE TABLE IF NOT EXISTS File_Process_Log (
    Run_ID VARCHAR, Source_Path VARCHAR, Source_File VARCHAR, File_Type VARCHAR,
    Campaign_ID VARCHAR, Month VARCHAR, Sheet_Name VARCHAR, Process_Status VARCHAR,
    Rows_Read INTEGER, Rows_Selected INTEGER, Extra_Columns INTEGER,
    Error_Message VARCHAR, Processing_Seconds DOUBLE, Processed_Time TIMESTAMP
)""")
ensure_columns("File_Process_Log", {
    "Rows_Selected": "INTEGER", "Extra_Columns": "INTEGER", "Processing_Seconds": "DOUBLE",
    "Rows_Other_Month": "INTEGER",   # valid date, but a different month  (expected, not an error)
    "Rows_Blank_Date": "INTEGER",    # date cell empty
    "Rows_Bad_Date": "INTEGER",      # date text could not be understood  (needs attention!)
    "Warning_Message": "VARCHAR",
})

# ---- Unreadable date values (so you can see exactly what to fix / add) ----
con.execute("""
CREATE TABLE IF NOT EXISTS Bad_Date_Log (
    Run_ID VARCHAR, Source_Path VARCHAR, File_Type VARCHAR, Month VARCHAR,
    Raw_Date VARCHAR, Row_Count INTEGER, Logged_Time TIMESTAMP
)""")

print("Monthly tables, File_Tracker, File_Process_Log and Bad_Date_Log are ready.")


Monthly tables, File_Tracker, File_Process_Log and Bad_Date_Log are ready.


In [8]:
# ==========================================
# OPTIONAL ONE-TIME RESET  (do NOT run during normal daily operation)
# ==========================================
# Rebuilds the CURRENT month from scratch and clears its tracker/log entries.
# Only touches the local database — never the source files.

def reset_current_month():
    con.execute("BEGIN TRANSACTION")
    try:
        con.execute(f'DROP TABLE IF EXISTS "{LIVE_TABLE}"')
        con.execute(f'DROP TABLE IF EXISTS "{WV_TABLE}"')
        con.execute("DELETE FROM File_Tracker WHERE Month = ?", [CURRENT_MONTH])
        con.execute("DELETE FROM File_Process_Log WHERE Month = ?", [CURRENT_MONTH])
        con.execute("DELETE FROM Bad_Date_Log WHERE Month = ?", [CURRENT_MONTH])
        con.execute("COMMIT")
        ensure_monthly_table(LIVE_TABLE, LIVE_TABLE_COLUMNS)
        ensure_monthly_table(WV_TABLE, WV_TABLE_COLUMNS)
        print(f"Reset complete for {CURRENT_MONTH}.")
    except Exception:
        con.execute("ROLLBACK")
        raise

# reset_current_month()


In [9]:
# ==========================================
# FILE DISCOVERY, METADATA, SOURCE FINGERPRINT  (read-only)
# ==========================================

def discover_files(folder):
    return sorted(
        f for f in folder.rglob("*.xlsx")
        if not f.name.startswith("~$")
    )


def get_file_metadata(file, source_folder):
    return {
        "Campaign_ID": file.parent.name,
        "Source_File": file.name,
        "Source_Path": file.relative_to(source_folder).as_posix(),
    }


def stat_file(file):
    st = file.stat()
    return datetime.fromtimestamp(st.st_mtime), st.st_size


def fingerprint(files):
    """(mtime_ns, size) for every source file — used to PROVE nothing was modified."""
    out = {}
    for f in files:
        st = f.stat()
        out[f.as_posix()] = (st.st_mtime_ns, st.st_size)
    return out


In [10]:
# ==========================================
# FORMAT-TOLERANT DATE ENGINE
# ==========================================
# Dates are used ONLY to decide "does this row belong to the target month?".
# The original text is stored unchanged. Nothing here changes your data.
#
# Handled: 2026-09-04 | 2026-09-04 10:32:11 | 2026-09-04T10:32:11Z | 04/09/2026 | 4/9/2026
#          04-09-2026 | 04.09.2026 | 04/09/26 | 4-Sep-2026 | 04-SEP-26 | Sep 4, 2026
#          September 4, 2026 | 4th Sep 2026 | 4 Sept 2026 | 20260904 | Excel serial numbers
#          (with or without time-of-day, AM/PM, time-zone)
#
# Ambiguity policy:  04/09/2026 is read per DAY_FIRST. If a value is impossible that way
# (09/13/2026) the other order is used automatically.  Any date outside 1990-2100 is
# treated as unreadable (reported, not trusted).

TIME_RE = r"[ T]\d{1,2}:\d{2}(:\d{2}(\.\d+)?)?\s*([AaPp][Mm])?\s*(Z|[+-]\d{2}:?\d{2})?\s*$"

CLEAN_TEMPLATE = (
    "trim(regexp_replace(regexp_replace(regexp_replace(trim(@RAW@), "
    "'" + TIME_RE + "', ''), "
    r"'(?i)(\d)(st|nd|rd|th)\b', '\1'), "
    r"'(?i)\bsept\b', 'Sep'))"
)

ISO_FORMATS            = ["%Y-%m-%d", "%Y/%m/%d", "%Y.%m.%d", "%Y%m%d"]
DAY_FIRST_FORMATS      = ["%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y"]
MONTH_FIRST_FORMATS    = ["%m/%d/%Y", "%m-%d-%Y", "%m.%d.%Y"]
NAMED_MONTH_FORMATS    = ["%d-%b-%Y", "%d %b %Y", "%d/%b/%Y", "%d-%B-%Y", "%d %B %Y",
                          "%b %d, %Y", "%B %d, %Y", "%b %d %Y", "%B %d %Y"]
SHORT_YEAR_DAY_FIRST   = ["%d/%m/%y", "%d-%m-%y", "%d.%m.%y"]
SHORT_YEAR_MONTH_FIRST = ["%m/%d/%y", "%m-%d-%y", "%m.%d.%y"]
SHORT_YEAR_NAMED       = ["%d-%b-%y", "%d %b %y", "%d-%B-%y"]


def _fmt_list(formats):
    return "[" + ", ".join("'" + f + "'" for f in formats) + "]"


def _ordered(day_first_group, month_first_group):
    return (day_first_group + month_first_group) if DAY_FIRST else (month_first_group + day_first_group)


def date_expr(raw):
    """SQL expression -> DATE (or NULL). `raw` is a SQL column reference."""
    clean = CLEAN_TEMPLATE.replace("@RAW@", raw)
    long_formats = _fmt_list(ISO_FORMATS + _ordered(DAY_FIRST_FORMATS, MONTH_FIRST_FORMATS) + NAMED_MONTH_FORMATS)
    short_formats = _fmt_list(_ordered(SHORT_YEAR_DAY_FIRST, SHORT_YEAR_MONTH_FIRST) + SHORT_YEAR_NAMED)
    num = f"TRY_CAST(trim({raw}) AS DOUBLE)"
    return (
        "CAST(COALESCE("
        f"CASE WHEN regexp_matches({clean}, '^\\d{{1,2}}[ /.-]([A-Za-z]+|\\d{{1,2}})[ /.-]\\d{{2}}$') "
        f"THEN try_strptime({clean}, {short_formats}) "
        f"ELSE try_strptime({clean}, {long_formats}) END, "
        f"CASE WHEN {num} BETWEEN 1 AND 60000 "
        f"THEN CAST(DATE '1899-12-30' + CAST(floor({num}) AS INTEGER) AS TIMESTAMP) END"
        ") AS DATE)"
    )


def safe_date_expr(raw):
    d = date_expr(raw)
    return f"CASE WHEN year({d}) BETWEEN 1990 AND 2100 THEN {d} END"


def ambiguous_expr(raw):
    """TRUE when a numeric slash/dash date lands in the target month under ONE reading but not the other."""
    clean = CLEAN_TEMPLATE.replace("@RAW@", raw)
    a = f"CAST(try_strptime({clean}, {_fmt_list(DAY_FIRST_FORMATS)}) AS DATE)"
    b = f"CAST(try_strptime({clean}, {_fmt_list(MONTH_FIRST_FORMATS)}) AS DATE)"
    in_a = f"({a} >= DATE '{MONTH_START}' AND {a} < DATE '{MONTH_END}')"
    in_b = f"({b} >= DATE '{MONTH_START}' AND {b} < DATE '{MONTH_END}')"
    return f"({a} IS NOT NULL AND {b} IS NOT NULL AND {a} <> {b} AND ({in_a} <> {in_b}))"


def test_date_parser(extra_cases=()):
    """Run this any time. Add your own (text, 'YYYY-MM-DD' or None) cases via extra_cases."""
    serial = (date(2026, 9, 4) - date(1899, 12, 30)).days
    cases = [
        ("2026-09-04", "2026-09-04"), ("2026-09-04 10:32:11", "2026-09-04"),
        ("2026-09-04T10:32:11Z", "2026-09-04"), ("04/09/2026", "2026-09-04" if DAY_FIRST else "2026-04-09"),
        ("13/09/2026", "2026-09-13"), ("09/13/2026", "2026-09-13"),
        ("4-Sep-2026", "2026-09-04"), ("04-SEP-26", "2026-09-04"), ("Sep 4, 2026", "2026-09-04"),
        ("4th Sep 2026", "2026-09-04"), ("04/09/26", "2026-09-04" if DAY_FIRST else "2026-04-09"),
        ("20260904", "2026-09-04"), (str(serial), "2026-09-04"), (f"{serial}.5432", "2026-09-04"),
        ("garbage", None), ("", None), ("31/02/2026", None),
    ] + list(extra_cases)
    t = duckdb.connect()
    t.execute("CREATE TABLE t(x VARCHAR)")
    t.executemany("INSERT INTO t VALUES (?)", [(c[0],) for c in cases])
    rows = t.execute(f"SELECT x, {safe_date_expr('x')} FROM t").fetchall()
    failures = 0
    for (text, expected), (_, got) in zip(cases, rows):
        got = got.isoformat() if got else None
        if got != expected:
            failures += 1
            print(f"FAIL  {text!r:32} -> {got}   (expected {expected})")
    print(f"Date parser self-test: {len(cases) - failures}/{len(cases)} passed.")
    return failures == 0


test_date_parser()


Date parser self-test: 17/17 passed.


True

In [11]:
# ==========================================
# SHEET DISCOVERY  (read-only; no heavy workbook load)
# ==========================================
# An .xlsx is a zip file. The sheet names live in one tiny XML file inside it,
# so we read only that — much cheaper than opening the whole workbook.

_NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"


def list_sheet_names(file):
    with zipfile.ZipFile(file, "r") as z:          # read-only
        root = ET.fromstring(z.read("xl/workbook.xml"))
    return [s.get("name") for s in root.iter(_NS + "sheet")]


def find_live_sheet(file):
    for name in list_sheet_names(file):
        if name.strip().lower() == "raw":
            return name
    return None


def find_wv_sheet(file):
    names = list_sheet_names(file)
    if len(names) != 1:
        raise ValueError(f"WV workbook must contain exactly 1 worksheet; found {len(names)}.")
    return names[0]


def sql_lit(text):
    return str(text).replace("'", "''")


def sql_ident(name):
    return '"' + str(name).replace('"', '""') + '"'


def source_relation(path, sheet):
    """The ONLY place where an Excel file is opened for data (read-only)."""
    return f"read_xlsx('{sql_lit(path.as_posix())}', sheet='{sql_lit(sheet)}', all_varchar=true)"


In [12]:
# ==========================================
# STAGE ONE FILE  (runs in a worker thread)
# ==========================================
# Read once -> map columns -> filter month -> return a small DataFrame.
# Workers NEVER write to the database and NEVER write to the source.

def stage_file(cur, file, file_type, source_folder):
    t0 = time.perf_counter()
    tmp_dir = None
    try:
        # ---- sheet ----
        if file_type == "LIVE":
            sheet = find_live_sheet(file)
            if sheet is None:
                raise ValueError("RAW sheet not found.")
            std_cols, cq_cols, date_col = LIVE_STANDARD_COLUMNS, LIVE_CQ_COLUMNS, "Date Created"
        else:
            sheet = find_wv_sheet(file)
            std_cols, cq_cols, date_col = WV_STANDARD_COLUMNS, WV_CQ_COLUMNS, "Date"

        # ---- optional read-only local copy (helps on slow network drives) ----
        read_path = file
        if COPY_TO_LOCAL:
            tmp_dir = Path(tempfile.mkdtemp(prefix="duckdb_stage_"))
            read_path = tmp_dir / file.name
            shutil.copyfile(file, read_path)            # reads source, writes only to temp

        # ---- read the sheet ONCE into a connection-private temp table ----
        cur.execute(f"CREATE OR REPLACE TEMP TABLE src AS SELECT * FROM {source_relation(read_path, sheet)}")
        source_headers = [r[0] for r in cur.execute("DESCRIBE src").fetchall()]
        if len(source_headers) == 0:
            raise ValueError(f"{file_type} file contains no columns.")

        if file_type == "LIVE":
            if len(source_headers) < len(live_required_columns):
                raise ValueError(
                    f"LIVE file has {len(source_headers)} columns; "
                    f"{len(live_required_columns)} are required."
                )
            header_results = build_live_header_results(source_headers)
            extra_columns = max(0, len(source_headers) - len(LIVE_STANDARD_COLUMNS))
        else:
            header_results = build_wv_header_results(source_headers)
            extra_columns = sum(1 for r in header_results.values() if r["type"] == "cq")

        mapping, cq_items = results_to_plan(source_headers, header_results)
        if date_col not in mapping:
            raise ValueError(f"{date_col} column not found.")

        # ---- build the SQL once per file (cost: O(columns), not O(rows)) ----
        select_std = ", ".join(
            f"{sql_ident(mapping[c])} AS {sql_ident(c)}" if c in mapping
            else f"CAST(NULL AS VARCHAR) AS {sql_ident(c)}"
            for c in std_cols
        )
        cq_parts = ", ".join(
            f"CASE WHEN trim({sql_ident(src)}) <> '' THEN '{sql_lit(prefix)}' || {sql_ident(src)} END"
            for src, prefix in cq_items
        )
        cq_list = f"list_filter([{cq_parts}]::VARCHAR[], x -> x IS NOT NULL)"

        raw = sql_ident(mapping[date_col])          # date column as it appears in `src`
        d = safe_date_expr(raw)
        in_month = f"({d} >= DATE '{MONTH_START}' AND {d} < DATE '{MONTH_END}')"

        # ---- date diagnostics (one pass over the temp table) ----
        n, blank, bad, ambiguous = cur.execute(f"""
            SELECT count(*),
                   count(*) FILTER (WHERE {raw} IS NULL OR trim({raw}) = ''),
                   count(*) FILTER (WHERE trim({raw}) <> '' AND {d} IS NULL),
                   count(*) FILTER (WHERE {ambiguous_expr(raw)})
            FROM src
        """).fetchone()

        bad_samples = []
        if bad:
            bad_samples = cur.execute(f"""
                SELECT trim({raw}), count(*) FROM src
                WHERE trim({raw}) <> '' AND {d} IS NULL
                GROUP BY 1 ORDER BY 2 DESC LIMIT {int(BAD_DATE_SAMPLE_LIMIT)}
            """).fetchall()

        # ---- the real work: map + CQ + month filter, all inside DuckDB ----
        cq_select = ", ".join(f"_cq[{i}] AS {sql_ident(c)}" for i, c in enumerate(cq_cols, start=1))
        std_select = ", ".join(sql_ident(c) for c in std_cols)
        df = cur.execute(f"""
            SELECT {std_select}, {cq_select}, len(_cq) AS _cq_n
            FROM (SELECT {select_std}, {cq_list} AS _cq FROM src) t
            WHERE {in_month}
        """).df()

        if len(df) and int(df["_cq_n"].max()) > len(cq_cols):
            raise ValueError(
                f"{file_type} file produced {int(df['_cq_n'].max())} CQ/exception values; "
                f"maximum supported is {len(cq_cols)}."
            )
        df = df.drop(columns="_cq_n")

        meta = get_file_metadata(file, source_folder)
        df["Campaign_ID"] = meta["Campaign_ID"]
        df["Source_File"] = meta["Source_File"]
        df["Source_Path"] = meta["Source_Path"]

        selected = len(df)
        return {
            "ok": True, "df": df, "sheet": sheet, "rows_read": n, "rows_selected": selected,
            "rows_blank_date": blank, "rows_bad_date": bad,
            "rows_other_month": n - selected - blank - bad,
            "ambiguous": ambiguous, "bad_samples": bad_samples,
            "extra_columns": extra_columns, "stage_seconds": time.perf_counter() - t0,
        }
    except Exception as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}", "stage_seconds": time.perf_counter() - t0}
    finally:
        try:
            cur.execute("DROP TABLE IF EXISTS src")
        except Exception:
            pass
        try:
            cur.close()
        except Exception:
            pass
        if tmp_dir is not None:
            shutil.rmtree(tmp_dir, ignore_errors=True)


In [13]:
# ==========================================
# TRACKER + LOG + LOAD HELPERS  (main thread only)
# ==========================================

def load_tracker():
    """Whole tracker for this month in ONE query -> dict (O(1) lookups afterwards)."""
    rows = con.execute("""
        SELECT Source_Path, File_Type, Last_Modified, Last_Size, Last_Status
        FROM File_Tracker WHERE Month = ?
    """, [CURRENT_MONTH]).fetchall()
    return {(r[0], r[1]): (r[2], r[3], r[4]) for r in rows}


def get_file_status(tracker, source_path, file_type, modified_time, size):
    prev = tracker.get((source_path, file_type))
    if prev is None:
        return "NEW"
    prev_modified, prev_size, prev_status = prev
    if prev_status == "FAILED":                      # (kept for old databases)
        return "CHANGED"
    if prev_modified is None or modified_time != prev_modified:
        return "CHANGED"
    if prev_size is not None and size != prev_size:  # same time but different size
        return "CHANGED"
    return "UNCHANGED"


def delete_previous_file_data(table_name, source_path):
    con.execute(f'DELETE FROM "{table_name}" WHERE Source_Path = ?', [source_path])


def insert_dataframe(df, table_name):
    """Bulk insert — one statement, not one per row."""
    if df.empty:
        return 0
    con.register("stg_df", df)
    try:
        con.execute(f'INSERT INTO "{table_name}" BY NAME SELECT * FROM stg_df')
    finally:
        con.unregister("stg_df")
    return len(df)


def upsert_file_tracker(source_path, source_file, file_type, campaign_id, month,
                        modified_time, size, run_id, status):
    con.execute("DELETE FROM File_Tracker WHERE Source_Path = ? AND File_Type = ? AND Month = ?",
                [source_path, file_type, month])
    con.execute("""
        INSERT INTO File_Tracker
        (Source_Path, Source_File, File_Type, Campaign_ID, Month,
         Last_Modified, Last_Size, Last_Run_ID, Last_Status)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [source_path, source_file, file_type, campaign_id, month,
          modified_time, size, run_id, status])


LOG_COLUMNS = ["Run_ID", "Source_Path", "Source_File", "File_Type", "Campaign_ID", "Month",
               "Sheet_Name", "Process_Status", "Rows_Read", "Rows_Selected", "Extra_Columns",
               "Rows_Other_Month", "Rows_Blank_Date", "Rows_Bad_Date", "Warning_Message",
               "Error_Message", "Processing_Seconds", "Processed_Time"]


def write_process_log(run_id, source_path, source_file, file_type, campaign_id, month,
                      sheet_name, process_status, rows_read, rows_selected, extra_columns,
                      error_message, processing_seconds,
                      rows_other_month=0, rows_blank_date=0, rows_bad_date=0, warning_message=None):
    cols = ", ".join(LOG_COLUMNS)
    marks = ", ".join("?" for _ in LOG_COLUMNS)
    con.execute(f"INSERT INTO File_Process_Log ({cols}) VALUES ({marks})", [
        run_id, source_path, source_file, file_type, campaign_id, month,
        sheet_name, process_status, rows_read, rows_selected, extra_columns,
        rows_other_month, rows_blank_date, rows_bad_date, warning_message,
        error_message, processing_seconds, datetime.now(),
    ])


def write_bad_dates(run_id, source_path, file_type, samples):
    if not samples:
        return
    now = datetime.now()
    con.executemany(
        "INSERT INTO Bad_Date_Log VALUES (?, ?, ?, ?, ?, ?, ?)",
        [(run_id, source_path, file_type, CURRENT_MONTH, v, c, now) for v, c in samples],
    )


def log_failure(run_id, item, error_text, seconds):
    try:
        con.execute("BEGIN TRANSACTION")
        write_process_log(run_id, item["source_path"], item["source_file"], item["file_type"],
                          item["campaign_id"], CURRENT_MONTH, None, "FAILED",
                          0, 0, 0, error_text, round(seconds, 3))
        con.execute("COMMIT")
    except Exception:
        try:
            con.execute("ROLLBACK")
        except Exception:
            pass


In [14]:
# ==========================================
# COMMIT ONE STAGED FILE  (delete + insert + tracker + log in ONE transaction)
# ==========================================

def commit_staged(run_id, item, staged):
    """If anything fails here, the transaction rolls back and OLD data stays intact."""
    table_name = LIVE_TABLE if item["file_type"] == "LIVE" else WV_TABLE
    t0 = time.perf_counter()

    warnings = []
    if staged["rows_bad_date"]:
        warnings.append(f"{staged['rows_bad_date']} row(s) have an UNREADABLE date (see Bad_Date_Log)")
    if staged["ambiguous"]:
        warnings.append(f"{staged['ambiguous']} row(s) have an AMBIGUOUS dd/mm vs mm/dd date "
                        f"(read as {'day-first' if DAY_FIRST else 'month-first'})")
    warning_message = "; ".join(warnings) if warnings else None

    con.execute("BEGIN TRANSACTION")
    try:
        if item["status"] == "CHANGED":
            delete_previous_file_data(table_name, item["source_path"])

        rows_inserted = insert_dataframe(staged["df"], table_name)
        total_seconds = round(staged["stage_seconds"] + (time.perf_counter() - t0), 3)

        upsert_file_tracker(item["source_path"], item["source_file"], item["file_type"],
                            item["campaign_id"], CURRENT_MONTH, item["modified"], item["size"],
                            run_id, item["status"])
        write_process_log(run_id, item["source_path"], item["source_file"], item["file_type"],
                          item["campaign_id"], CURRENT_MONTH, staged["sheet"], item["status"],
                          staged["rows_read"], staged["rows_selected"], staged["extra_columns"],
                          None, total_seconds,
                          staged["rows_other_month"], staged["rows_blank_date"],
                          staged["rows_bad_date"], warning_message)
        write_bad_dates(run_id, item["source_path"], item["file_type"], staged["bad_samples"])
        con.execute("COMMIT")
    except Exception:
        try:
            con.execute("ROLLBACK")
        except Exception:
            pass
        raise

    flag = "  !! " + warning_message if warning_message else ""
    print(f"{item['status']:9} | {item['file_type']:4} | {item['source_path']} | "
          f"read={staged['rows_read']} | selected={staged['rows_selected']} | "
          f"other_month={staged['rows_other_month']} | blank_date={staged['rows_blank_date']} | "
          f"bad_date={staged['rows_bad_date']} | {total_seconds}s{flag}")
    return {"status": item["status"], "rows": rows_inserted,
            "bad_dates": staged["rows_bad_date"], "ambiguous": staged["ambiguous"]}


In [15]:
# ==========================================
# REFRESH SUMMARY + FINAL RUNNER
# ==========================================

def create_refresh_summary(results):
    summary = {"NEW": 0, "CHANGED": 0, "UNCHANGED": 0, "FAILED": 0, "ROWS": 0,
               "BAD_DATE_ROWS": 0, "AMBIGUOUS_DATE_ROWS": 0}
    for r in results:
        if r["status"] in summary:
            summary[r["status"]] += 1
        summary["ROWS"] += r["rows"]
        summary["BAD_DATE_ROWS"] += r.get("bad_dates", 0)
        summary["AMBIGUOUS_DATE_ROWS"] += r.get("ambiguous", 0)
    return summary


def run_project(max_workers=None):
    max_workers = max_workers or MAX_WORKERS
    run_started = time.perf_counter()
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3]   # includes milliseconds -> always unique

    print("=" * 78)
    print("DUCKDB PROJECT REFRESH  (v2)")
    print("=" * 78)
    print("Run ID  :", run_id)
    print("Month   :", CURRENT_MONTH)
    print("Source  :", SOURCE_MONTH_FOLDER)
    print("DB      :", DB_FILE)
    print("Workers :", max_workers, "| copy-to-local:", COPY_TO_LOCAL)
    print()

    ensure_monthly_table(LIVE_TABLE, LIVE_TABLE_COLUMNS)
    ensure_monthly_table(WV_TABLE, WV_TABLE_COLUMNS)

    live_files = discover_files(LIVE_FOLDER)
    wv_files = discover_files(WV_FOLDER)
    print("LIVE files found :", len(live_files))
    print("WV files found   :", len(wv_files))
    print()

    before = fingerprint(live_files + wv_files)          # source fingerprint (read-only stat)
    tracker = load_tracker()

    results, work, unchanged_rows = [], [], []

    # ---- 1. Decide NEW / CHANGED / UNCHANGED without opening any Excel file ----
    for file_type, files, folder in (("LIVE", live_files, LIVE_FOLDER), ("WV", wv_files, WV_FOLDER)):
        for file in files:
            meta = get_file_metadata(file, folder)
            modified, size = stat_file(file)
            status = get_file_status(tracker, meta["Source_Path"], file_type, modified, size)
            item = {"file": file, "file_type": file_type, "folder": folder,
                    "source_path": meta["Source_Path"], "source_file": meta["Source_File"],
                    "campaign_id": meta["Campaign_ID"], "modified": modified, "size": size,
                    "status": status}
            if status == "UNCHANGED":
                results.append({"status": "UNCHANGED", "rows": 0})
                unchanged_rows.append((run_id, item["source_path"], item["source_file"], file_type,
                                       item["campaign_id"], CURRENT_MONTH, None, "UNCHANGED",
                                       0, 0, 0, 0, 0, 0, None, None, 0.0, datetime.now()))
                print(f"UNCHANGED | {file_type} | {item['source_path']}")
            else:
                work.append(item)

    # ---- 2. One batched log write for all UNCHANGED files ----
    if unchanged_rows:
        con.execute("BEGIN TRANSACTION")
        try:
            cols = ", ".join(LOG_COLUMNS)
            marks = ", ".join("?" for _ in LOG_COLUMNS)
            con.executemany(f"INSERT INTO File_Process_Log ({cols}) VALUES ({marks})", unchanged_rows)
            con.execute("COMMIT")
        except Exception:
            con.execute("ROLLBACK")
            raise

    # ---- 3. NEW / CHANGED: read in parallel, write one by one ----
    if work:
        print()
        print("-" * 78)
        print(f"PROCESSING {len(work)} NEW/CHANGED FILE(S)")
        print("-" * 78)

    batch_size = max_workers * 3          # keeps memory bounded
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        for start in range(0, len(work), batch_size):
            batch = work[start:start + batch_size]
            futures = {}
            for item in batch:
                cur = con.cursor()        # one private connection per file (created on main thread)
                fut = pool.submit(stage_file, cur, item["file"], item["file_type"], item["folder"])
                futures[fut] = item

            for fut in as_completed(futures):
                item = futures[fut]
                staged = fut.result()
                if not staged["ok"]:
                    log_failure(run_id, item, staged["error"], staged["stage_seconds"])
                    print(f"FAILED    | {item['file_type']:4} | {item['source_path']} | {staged['error']}")
                    results.append({"status": "FAILED", "rows": 0})
                    continue
                try:
                    results.append(commit_staged(run_id, item, staged))
                except Exception as e:
                    log_failure(run_id, item, f"{type(e).__name__}: {e}", staged["stage_seconds"])
                    print(f"FAILED    | {item['file_type']:4} | {item['source_path']} | {type(e).__name__}: {e}")
                    results.append({"status": "FAILED", "rows": 0})

    summary = create_refresh_summary(results)

    # ---- 4. Prove the source was not touched ----
    after = fingerprint(live_files + wv_files)
    changed_during_run = [p for p in before if after.get(p) != before[p]]
    summary["SOURCE_CHANGED"] = len(changed_during_run)

    elapsed = round(time.perf_counter() - run_started, 2)
    print()
    print("=" * 78)
    print("REFRESH COMPLETE")
    print("=" * 78)
    print("NEW                 :", summary["NEW"])
    print("CHANGED             :", summary["CHANGED"])
    print("UNCHANGED           :", summary["UNCHANGED"])
    print("FAILED              :", summary["FAILED"])
    print("ROWS INSERTED       :", summary["ROWS"])
    print("BAD-DATE ROWS       :", summary["BAD_DATE_ROWS"], "  <- check Bad_Date_Log if > 0")
    print("AMBIGUOUS-DATE ROWS :", summary["AMBIGUOUS_DATE_ROWS"], "  <- dd/mm vs mm/dd, review if > 0")
    if changed_during_run:
        print("Source integrity    : WARNING - these files changed while the run was in progress.")
        print("                      (This notebook only reads; someone else saved them.)")
        for p in changed_during_run:
            print("                      ", p)
    else:
        print("Source integrity    : OK (no source file was modified)")
    print("Total time          :", elapsed, "seconds")
    print("=" * 78)
    return summary


## Normal daily operation

```python
summary = run_project()
```

When the month changes, update **only** `SOURCE_MONTH_FOLDER` (the month label is derived from it).

- `UNCHANGED` files are skipped without opening Excel.
- `Bad_Date_Log` shows any date text the engine could not read — fix the source value, or add the new format to the lists in the date-engine cell. Run `test_date_parser()` after adding one.
- Do **not** run `reset_current_month()` during normal operation.


In [16]:
summary = run_project()

DUCKDB PROJECT REFRESH  (v2)
Run ID  : 20261006_114353_375
Month   : Sep26
Source  : C:\Users\Abhay\OneDrive\Desktop\DuckDB\2026\Sep
DB      : C:\Users\Abhay\OneDrive\Desktop\Claude DuckDB result\company_data.duckdb
Workers : 4 | copy-to-local: False

LIVE files found : 6
WV files found   : 6


------------------------------------------------------------------------------
PROCESSING 12 NEW/CHANGED FILE(S)
------------------------------------------------------------------------------
NEW       | LIVE | 10726065_Bloomberg Singapore/QA Report 10726065_Bloomberg Singapore Pte LtdBloomberg Treasury ABM Thailand - D7B1EDLinked.xlsx | read=38 | selected=11 | other_month=27 | blank_date=0 | bad_date=0 | 1.236s
NEW       | LIVE | 10626044_Syntax Systems LimitedSyntax FY26Q2/QA Report 10626044_Syntax Systems LimitedSyntax FY26Q2 DACH LGM CampaignLinked_E1CE6 Sep'26.xlsx | read=117 | selected=27 | other_month=90 | blank_date=0 | bad_date=0 | 1.151s
NEW       | LIVE | 10626024_TomTom Global Conten

In [17]:
# ==========================================
# VERIFY: does the database match what the log says was loaded?
# ==========================================

def verify_month():
    q = f"""
    WITH last_ok AS (
        SELECT File_Type, Source_Path, Rows_Selected, Rows_Read, Rows_Other_Month,
               Rows_Blank_Date, Rows_Bad_Date
        FROM File_Process_Log
        WHERE Month = ? AND Process_Status IN ('NEW', 'CHANGED')
        QUALIFY row_number() OVER (PARTITION BY File_Type, Source_Path ORDER BY Processed_Time DESC) = 1
    ),
    actual AS (
        SELECT 'LIVE' AS File_Type, Source_Path, count(*) AS In_Table FROM "{LIVE_TABLE}" GROUP BY Source_Path
        UNION ALL
        SELECT 'WV', Source_Path, count(*) FROM "{WV_TABLE}" GROUP BY Source_Path
    )
    SELECT l.File_Type, l.Source_Path, l.Rows_Read, l.Rows_Selected,
           coalesce(a.In_Table, 0) AS In_Table,
           l.Rows_Other_Month, l.Rows_Blank_Date, l.Rows_Bad_Date,
           CASE WHEN l.Rows_Selected = coalesce(a.In_Table, 0) THEN 'OK' ELSE 'MISMATCH' END AS Check_Result
    FROM last_ok l LEFT JOIN actual a USING (File_Type, Source_Path)
    ORDER BY (Check_Result = 'OK'), l.File_Type, l.Source_Path
    """
    df = con.execute(q, [CURRENT_MONTH]).df()
    bad = int((df["Check_Result"] != "OK").sum()) if len(df) else 0
    print(f"{len(df)} file(s) checked | {bad} mismatch(es)")
    return df

verify_month()


10 file(s) checked | 0 mismatch(es)


,File_Type,Source_Path,Rows_Read,Rows_Selected,In_Table,Rows_Other_Month,Rows_Blank_Date,Rows_Bad_Date,Check_Result
0,LIVE,10626024_TomTom Global Content/QA Report 10626...,228,128,128,100,0,0,OK
1,LIVE,10626043_Syntax Systems/QA Report 10626043_Syn...,370,68,68,302,0,0,OK
2,LIVE,10626044_Syntax Systems LimitedSyntax FY26Q2/Q...,117,27,27,90,0,0,OK
3,LIVE,10726065_Bloomberg Singapore/QA Report 1072606...,38,11,11,27,0,0,OK
4,LIVE,10726094_TechzineTechzine/QA Report 10726094_T...,100,17,17,83,0,0,OK
5,LIVE,12812526_TechzineTechzine/QA Report 12812526_T...,101,18,18,83,0,0,OK
6,WV,10626022_TomTom Global Content B.V TomTom FY2/...,575,413,413,162,0,0,OK
7,WV,12812526 Abhay cheking new/WV Report - 1281252...,84,84,84,0,0,0,OK
8,WV,12812526 Abhay cheking new/WV Report - 1281252...,84,84,84,0,0,0,OK
9,WV,WV_10626005_Customer Management Practice Custo...,676,294,294,382,0,0,OK


In [18]:
# ==========================================
# HANDY QUERIES
# ==========================================

print(con.sql("SHOW TABLES").fetchall())

# Files that FAILED in the latest run (and why)
con.sql("""
    SELECT Source_Path, File_Type, Error_Message, Processed_Time
    FROM File_Process_Log
    WHERE Month = '%s' AND Process_Status = 'FAILED'
    ORDER BY Processed_Time DESC
""" % CURRENT_MONTH).df()


[('Bad_Date_Log',), ('File_Process_Log',), ('File_Tracker',), ('LIVE_Sep26',), ('WV_Sep26',)]


,Source_Path,File_Type,Error_Message,Processed_Time
0,WV_10626005_Customer Management Practice Custo...,WV,PermissionError: [Errno 13] Permission denied:...,2026-10-06 11:43:55.948095
1,10626004 - Customer Management PracticeCust/WV...,WV,ValueError: WV workbook must contain exactly 1...,2026-10-06 11:43:55.185046


In [19]:
# Unreadable date values saved by the latest runs  (empty = great)
con.sql("""
    SELECT Source_Path, Raw_Date, Row_Count
    FROM Bad_Date_Log
    WHERE Month = '%s'
    ORDER BY Logged_Time DESC, Row_Count DESC
""" % CURRENT_MONTH).df()


,Source_Path,Raw_Date,Row_Count


In [20]:
# Look up one email (parameterised — no editing of SQL needed)
def lookup_email(email):
    return con.execute(
        f'SELECT * FROM "{LIVE_TABLE}" WHERE lower(trim("Email ID")) = lower(trim(?))', [email]
    ).df()

# lookup_email("matti.korvenmaa@swarco.com")


In [25]:
# ==========================================
# QUALIFIED LEADS -> EXCEL  (output goes to the PROJECT folder, never the source)
# ==========================================

QUALIFIED_SQL = f"""
    SELECT * FROM "{LIVE_TABLE}"
    WHERE lower(trim("QA Status")) = 'qualified'
"""

def export_query_to_excel(sql, output_file):
    """Fast path: DuckDB writes the .xlsx itself. Falls back to pandas if unavailable."""
    output_file = Path(output_file)
    try:
        con.execute("INSTALL excel")
        con.execute("LOAD excel")
        con.execute(f"COPY ({sql}) TO '{sql_lit(output_file.as_posix())}' (FORMAT xlsx, HEADER true)")
        method = "DuckDB COPY"
    except Exception as e:
        print("DuckDB xlsx export not available, using pandas instead:", type(e).__name__)
        con.execute(sql).df().to_excel(output_file, index=False)
        method = "pandas"
    print(f"Exported via {method}: {output_file}")

qualified_count = con.execute(f"SELECT count(*) FROM ({QUALIFIED_SQL})").fetchone()[0]
print("Qualified leads:", qualified_count)

# Run when needed:
# export_query_to_excel(QUALIFIED_SQL, PROJECT_FOLDER / f"{LIVE_TABLE}_Qualified_{datetime.now():%d_%m}.xlsx")


Qualified leads: 224


In [23]:
con.sql("select count(*) from wv_sep26")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          875 │
└──────────────┘

In [26]:
# ==========================================
# EXPORT LOG TABLE(S) TO EXCEL
# ==========================================

def export_logs_to_excel(only_current_month=False):
    output_file = PROJECT_FOLDER / f"Process_Log_{datetime.now():%Y%m%d_%H%M%S}.xlsx"

    where = f"WHERE Month = '{CURRENT_MONTH}'" if only_current_month else ""

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sheet, table in [
            ("File_Process_Log", "File_Process_Log"),
            ("Bad_Date_Log",     "Bad_Date_Log"),
            ("File_Tracker",     "File_Tracker"),
        ]:
            df = con.execute(f"SELECT * FROM {table} {where}").df()
            df.to_excel(writer, sheet_name=sheet, index=False)

            # make the header row readable: freeze it and widen the columns
            ws = writer.sheets[sheet]
            ws.freeze_panes = "A2"
            for col in ws.columns:
                longest = max((len(str(c.value)) for c in col if c.value is not None), default=10)
                ws.column_dimensions[col[0].column_letter].width = min(longest + 2, 60)

            print(f"{sheet:18} -> {len(df):,} rows")

    print("Exported:", output_file)

export_logs_to_excel()                        # everything, all months
# export_logs_to_excel(only_current_month=True)   # only Sep26

File_Process_Log   -> 12 rows
Bad_Date_Log       -> 0 rows
File_Tracker       -> 10 rows
Exported: C:\Users\Abhay\OneDrive\Desktop\Claude DuckDB result\Process_Log_20261006_115715.xlsx


In [27]:
# ==========================================
# EXPORT LIVE / WV TABLE TO EXCEL
# ==========================================

EXCEL_MAX_ROWS = 1_048_575      # Excel's limit per sheet (1,048,576 minus the header row)

def export_table_to_excel(table_name, label=None, source_file=None, max_rows=EXCEL_MAX_ROWS):
    label = label or table_name
    where, params = "", []
    if source_file:                                   # optional: export just one source file
        where, params = "WHERE Source_File = ?", [source_file]

    total = con.execute(f'SELECT count(*) FROM "{table_name}" {where}', params).fetchone()[0]
    if total == 0:
        print(f"{table_name}: no rows to export.")
        return None

    output_file = PROJECT_FOLDER / f"{label}_{datetime.now():%Y%m%d_%H%M%S}.xlsx"
    parts = (total + max_rows - 1) // max_rows        # more than 1 only if the table exceeds Excel's limit

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for part in range(parts):
            df = con.execute(
                f'SELECT * FROM "{table_name}" {where} OFFSET {part * max_rows} LIMIT {max_rows}',
                params
            ).df()
            sheet = label[:28] if parts == 1 else f"{label[:24]}_{part + 1}"
            df.to_excel(writer, sheet_name=sheet, index=False)

            ws = writer.sheets[sheet]
            ws.freeze_panes = "A2"                    # header stays visible
            ws.auto_filter.ref = ws.dimensions        # filter buttons on every column
            for col in ws.iter_cols(min_row=1, max_row=min(len(df) + 1, 500)):
                longest = max((len(str(c.value)) for c in col if c.value is not None), default=10)
                ws.column_dimensions[col[0].column_letter].width = min(longest + 2, 50)

            print(f"{table_name}: sheet '{sheet}' -> {len(df):,} rows")

    print("Exported:", output_file)
    return output_file


export_table_to_excel(LIVE_TABLE)      # whole LIVE_Sep26
export_table_to_excel(WV_TABLE)        # whole WV_Sep26

LIVE_Sep26: sheet 'LIVE_Sep26' -> 269 rows
Exported: C:\Users\Abhay\OneDrive\Desktop\Claude DuckDB result\LIVE_Sep26_20261006_120115.xlsx
WV_Sep26: sheet 'WV_Sep26' -> 875 rows
Exported: C:\Users\Abhay\OneDrive\Desktop\Claude DuckDB result\WV_Sep26_20261006_120116.xlsx


WindowsPath('C:/Users/Abhay/OneDrive/Desktop/Claude DuckDB result/WV_Sep26_20261006_120116.xlsx')

In [32]:
import os, stat
from pathlib import Path

bad = Path(r"C:\Users\Abhay\OneDrive\Desktop\DuckDB\2026\Sep\WV\WV_10626005_Customer Management Practice Customer Management.xlsx")

print("1. Folder exists   :", bad.parent.exists())
print("2. File exists     :", bad.exists(), "| is a file:", bad.is_file(), "| is a folder:", bad.is_dir())

# Similar names in the same folder (catches typos, double spaces, trailing spaces)
if bad.parent.exists():
    similar = [p.name for p in bad.parent.glob("*10626005*")]
    print("3. Names containing 10626005:")
    for n in similar:
        print("     ", repr(n))
    locks = [p.name for p in bad.parent.glob("~$*")]
    print("4. Excel lock files in folder (file open somewhere):", locks if locks else "none")

if bad.exists() and bad.is_file():
    st = bad.stat()
    attrs = getattr(st, "st_file_attributes", 0)          # Windows only
    print("5. Size (bytes)    :", st.st_size)
    print("   Read-only flag  :", bool(attrs & stat.FILE_ATTRIBUTE_READONLY) if attrs else "n/a")
    print("   OneDrive cloud-only (not downloaded):",
          bool(attrs & (0x400000 | 0x40000 | 0x1000)) if attrs else "n/a")
    try:
        with open(bad, "rb") as f:
            f.read(1024)
        print("6. Read test       : OK  -> Python CAN read it now. Just re-run run_project().")
    except Exception as e:
        print("6. Read test       : FAILED ->", type(e).__name__, "-", e)

1. Folder exists   : True
2. File exists     : True | is a file: False | is a folder: True
3. Names containing 10626005:
      'WV_10626005_Customer Management Practice Customer Management.xlsx'
4. Excel lock files in folder (file open somewhere): none


In [33]:
bad = Path(r"C:\Users\Abhay\OneDrive\Desktop\DuckDB\2026\Sep\WV\WV_10626005_Customer Management Practice Customer Management.xlsx")

print("Exists   :", bad.exists())
try:
    with open(bad, "rb") as f:
        f.read(1024)
    print("Read test: OK, the file can be read now")
except Exception as e:
    print("Read test: FAILED ->", type(e).__name__, e)

lock = bad.parent / ("~$" + bad.name[2:])   # Excel names its lock file this way
print("Excel lock file present:", lock.exists() or any(bad.parent.glob("~$*")))

Exists   : True
Read test: FAILED -> PermissionError [Errno 13] Permission denied: 'C:\\Users\\Abhay\\OneDrive\\Desktop\\DuckDB\\2026\\Sep\\WV\\WV_10626005_Customer Management Practice Customer Management.xlsx'
Excel lock file present: False
